# Inspect Aggregated AET Datasets

HRU-level inspection of the two AET sources. Mirrors `inspect_consolidated_aet.ipynb`.

Sources (see `catalog/variables.yml` → `aet`):

- SSEBop (`et`, mm/month) — accessed via USGS NHGF STAC; consolidated and aggregated.
- MOD16A2 v061 (`ET_500m`, kg m⁻² per 8-day composite — `scale_factor=0.1`) — global tiles consolidated to CONUS+ then aggregated.

See `docs/references/calibration-target-recipes.md` §2 for canonical-unit conversions and the open `scale_factor` question.

## Per-target conventions in this notebook

- SSEBop `et` is native mm/month. No conversion before plotting or comparison.
- MOD16A2 `ET_500m` is kg m⁻² per 8-day composite, with `scale_factor=0.1` per the HDF spec. We apply the scale factor explicitly in cell 9 to defend against the missing-`decode_cf` case (recipes §2 — flagged 500× domain-mean offset in the consolidated notebook).
- Per-month aggregation of MOD16A2 8-day composites: weight each composite by `(days of overlap with target month) / 8`, then sum weighted contributions for all composites that intersect the month. **For the inspection notebook we use the simpler approximation: take the composite whose start lies in the target month and scale to mm/month.** This is good enough for spot-checking; the target builder will do the proper overlap weighting.
- Reference source: SSEBop.

In [ ]:
import calendar
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from _helpers import (
    area_weighted_mean,
    discover_aggregated,
    load_fabric,
    load_project_paths,
    lookup_hrus_by_points,
    nan_hru_count,
    open_year,
    open_year_range,
    plot_hru_choropleth,
    plot_nan_hrus,
    save_figure,
    select_month,
    unit_from_catalog,
)

# Edit me to point at a real project directory:
PROJECT_DIR = Path(
    "/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets"
)

# Set True (and re-run) to populate docs/figures/inspect_aggregated/<run>_*.png
# import _helpers
# _helpers.SAVE_FIGURES = True

project_dir, datastore_dir, fabric_cfg = load_project_paths(PROJECT_DIR)
fabric = load_fabric(fabric_cfg)

TARGET = "aet"
TARGET_TIME = "2000-01-15"
TARGET_YEAR = 2000
TARGET_MONTH = 1
TIME_SERIES_YEARS = range(2000, 2011)
REPRESENTATIVE_POINTS = {
    "Olympic Peninsula (PNW mountains)": (-123.5, 47.8),
    "Iowa cropland (Midwest)": (-93.6, 42.0),
    "Phoenix metro (arid SW)": (-112.1, 33.4),
    "Southern Appalachians (Eastern forest)": (-83.5, 35.5),
}

SOURCES = {
    "ssebop": {"label": "SSEBop (et)", "var": "et"},
    "mod16a2_v061": {"label": "MOD16A2 v061 (ET_500m)", "var": "ET_500m"},
}

print(f"Project: {project_dir}")
print(f"Datastore: {datastore_dir}")
print(f"Fabric: {fabric_cfg['path']} ({len(fabric)} HRUs)")

## Load aggregated datasets

Each source is opened from `<project>/data/aggregated/<source_key>/<source_key>_<TARGET_YEAR>_agg.nc`. Sources whose aggregation has not yet been produced are skipped with a clear message; downstream cells iterate over the loaded set so missing sources drop out naturally.

In [ ]:
opened = {}
for source_key, info in SOURCES.items():
    paths = discover_aggregated(project_dir, source_key)
    if paths is None:
        print(f"SKIP {info['label']}: no aggregated NCs at "
              f"{project_dir}/data/aggregated/{source_key}/")
        continue
    ds = open_year(project_dir, source_key, TARGET_YEAR)
    info["units"] = unit_from_catalog(source_key, info["var"])
    opened[source_key] = (ds, info)
    values = ds[info["var"]].isel(time=0).to_pandas()
    print(
        f"{info['label']}: vars={list(ds.data_vars)} | "
        f"time={ds.time.values[0]}..{ds.time.values[-1]} | "
        f"HRUs={ds.sizes.get('nhm_id', ds.sizes.get('hru_id', 'unknown'))} | "
        f"NaN HRUs (t=0): {nan_hru_count(values)} | "
        f"catalog units: {info['units']}"
    )

In [ ]:
for source_key, (ds, info) in opened.items():
    print(f"{'=' * 60}\n{info['label']}\n{'=' * 60}")
    display(ds)

In [ ]:
if not opened:
    print("No sources available; skipping native-unit map.")
else:
    fig, axes = plt.subplots(1, len(opened), figsize=(8 * len(opened), 6), squeeze=False)
    for idx, (source_key, (ds, info)) in enumerate(opened.items()):
        da = select_month(ds[info["var"]], TARGET_YEAR, TARGET_MONTH)
        values = da.to_pandas()
        plot_hru_choropleth(
            axes[0, idx],
            fabric,
            values,
            cmap="YlGnBu",
            title=f"{info['label']}\n{TARGET_TIME} | {info['units']}",
            units=info["units"],
        )
    fig.suptitle(f"AET — native units, {TARGET_TIME}", fontsize=13, y=1.02)
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_native_units_map")
    plt.show()

## Normalized comparison map (mm/month)

Convert both sources to **mm/month** and plot side by side on a shared colour scale derived from SSEBop (the reference source).

- SSEBop `et` is native mm/month — no conversion.
- MOD16A2 v061 `ET_500m` (kg m⁻² per 8-day composite) × `scale_factor = 0.1` → mm per composite. For monthly comparison we use the simpler approximation: take the composite that starts in the target month, scale by 0.1, and treat as mm/month. The target builder will apply proper overlap weighting (recipes §2).

Both panels are on the HRU fabric, so the geographic footprint is identical; the colour scale is anchored on SSEBop's distribution.


In [ ]:
def _to_mm_per_month(da: xr.DataArray, source_key: str) -> xr.DataArray:
    if source_key == "ssebop":
        return da
    if source_key == "mod16a2_v061":
        # scale_factor=0.1 per MOD16A2 v061 HDF spec; defend against
        # missing decode_cf on the consolidated NC (recipes §2)
        return da * 0.1
    raise ValueError(f"No mm/month conversion for {source_key}")


if opened:
    converted = {}
    for source_key, (ds, info) in opened.items():
        da = select_month(ds[info["var"]], TARGET_YEAR, TARGET_MONTH)
        converted[source_key] = _to_mm_per_month(da, source_key).to_pandas()

    ref_key = "ssebop"
    if ref_key in converted:
        ref_vals = converted[ref_key].dropna().values
        vmin, vmax = float(np.percentile(ref_vals, 2)), float(np.percentile(ref_vals, 98))
    else:
        all_vals = np.concatenate([s.dropna().values for s in converted.values()])
        vmin, vmax = float(np.percentile(all_vals, 2)), float(np.percentile(all_vals, 98))

    fig, axes = plt.subplots(1, len(converted), figsize=(8 * len(converted), 6), squeeze=False)
    for idx, (source_key, values) in enumerate(converted.items()):
        plot_hru_choropleth(
            axes[0, idx],
            fabric,
            values,
            vmin=vmin,
            vmax=vmax,
            cmap="YlGnBu",
            title=f"{SOURCES[source_key]['label']}\n{TARGET_TIME} | mm/month",
            units="mm/month",
        )
    fig.suptitle(
        f"AET — mm/month, colour scale from SSEBop — {TARGET_TIME}",
        fontsize=13, y=1.02,
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_normalized_comparison")
    plt.show()

In [ ]:
if opened:
    fig, ax = plt.subplots(figsize=(10, 5))
    for source_key, values in converted.items():
        ax.hist(
            values.dropna(),
            bins=60,
            histtype="step",
            label=SOURCES[source_key]["label"],
            density=True,
            linewidth=2,
        )
    ax.set_xlabel("AET (mm/month)")
    ax.set_ylabel("Density")
    ax.set_title(f"Cross-source HRU distribution — {TARGET_TIME}")
    ax.legend()
    save_figure(fig, f"{TARGET}_histogram")
    plt.show()

In [ ]:
if opened:
    rep_hrus = lookup_hrus_by_points(fabric, REPRESENTATIVE_POINTS)
    print("Representative HRUs:", rep_hrus)

    series_data = {}
    for source_key, info in SOURCES.items():
        if source_key not in opened:
            continue
        ds_range = open_year_range(project_dir, source_key, TIME_SERIES_YEARS)
        try:
            id_dim = "nhm_id" if "nhm_id" in ds_range.dims else "hru_id"
            sel = ds_range[info["var"]].sel({id_dim: list(rep_hrus.values())}).load()
        finally:
            ds_range.close()
        series_data[source_key] = sel

    def _convert_series(da, source_key):
        """Apply per-source unit conversion. Expects 1-D (time,) input — select an HRU first."""
        if source_key == "ssebop":
            return da
        if source_key == "mod16a2_v061":
            return da * 0.1
        return da

    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
    id_dim = "nhm_id" if "nhm_id" in next(iter(series_data.values())).dims else "hru_id"
    for ax, (label, hru_id) in zip(axes.flat, rep_hrus.items()):
        for source_key, da in series_data.items():
            da_hru = _convert_series(da.sel({id_dim: hru_id}), source_key)
            ax.plot(da_hru.time, da_hru.values, label=SOURCES[source_key]["label"])
        ax.set_title(f"{label} (HRU {hru_id})")
        ax.set_ylabel("mm/month")
        ax.legend(fontsize=8)
    fig.suptitle(f"AET at representative HRUs — {min(TIME_SERIES_YEARS)}–{max(TIME_SERIES_YEARS)}")
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_time_series")
    plt.show()

In [ ]:
if opened:
    fig, axes = plt.subplots(1, len(opened), figsize=(8 * len(opened), 6), squeeze=False)
    for idx, (source_key, (ds, info)) in enumerate(opened.items()):
        da = select_month(ds[info["var"]], TARGET_YEAR, TARGET_MONTH)
        values = da.to_pandas()
        n_nan = nan_hru_count(values)
        print(f"{info['label']}: {n_nan} NaN HRUs ({100 * n_nan / len(fabric):.2f}%)")
        plot_nan_hrus(
            axes[0, idx],
            fabric,
            values,
            title=f"{info['label']}\nNaN HRUs (red) — {n_nan} of {len(fabric)}",
        )
    fig.suptitle(
        "Coverage gaps — these will be nearest-neighbor-filled in normalize/ before target combination",
        fontsize=12,
        y=1.02,
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_coverage")
    plt.show()

In [ ]:
def _gridded_mean_aet(source_key, info):
    """Compute the gridded CONUS-footprint mean for the validation cell.

    Approximate cross-check, not a like-for-like comparison: gridded mean
    is unweighted over the consolidated-NC bbox (which extends past the
    fabric into ocean/Canada/Mexico edges); HRU mean is Albers-area-
    weighted over fabric HRUs only. Differences of 5-30% are normal for
    sources with significant out-of-fabric coverage.
    """
    if source_key == "ssebop":
        # SSEBop is remote zarr accessed via the NHGF STAC.
        # If the project's network access doesn't allow STAC reads at notebook
        # time, skip-with-reason.
        try:
            import fsspec
            store = "s3://mdmf/gdp/ssebopeta_monthly.zarr/"
            ds = xr.open_zarr(
                fsspec.get_mapper(
                    store,
                    anon=True,
                    client_kwargs={"endpoint_url": "https://usgs.osn.mghpcc.org/"},
                ),
                consolidated=True,
            )
        except Exception as exc:
            return None, f"STAC fetch failed: {exc}"
        try:
            da = select_month(ds[info["var"]], TARGET_YEAR, TARGET_MONTH).load()
        finally:
            ds.close()
        return _to_mm_per_month(da, source_key).mean(skipna=True).item(), None
    if source_key == "mod16a2_v061":
        path = datastore_dir / "mod16a2_v061" / f"mod16a2_v061_{TARGET_YEAR}.nc"
        if not path.exists():
            return None, f"missing consolidated NC at {path}"
        with xr.open_dataset(path) as ds:
            da = select_month(ds[info["var"]], TARGET_YEAR, TARGET_MONTH).load()
        return _to_mm_per_month(da, source_key).mean(skipna=True).item(), None
    return None, f"unknown gridded path for {source_key}"


print(f"{'Source':<35} {'Aggregated':>12} {'Gridded':>12} {'Δ':>12} {'% diff':>8}")
print("-" * 85)
for source_key, (ds, info) in opened.items():
    da_agg = select_month(ds[info["var"]], TARGET_YEAR, TARGET_MONTH)
    converted_agg = _to_mm_per_month(da_agg, source_key).to_pandas()
    agg_mean = area_weighted_mean(converted_agg, fabric)
    gridded_mean, reason = _gridded_mean_aet(source_key, info)
    if gridded_mean is None:
        print(f"{info['label']:<35} {agg_mean:>12.3f}  {'SKIP':>12} ({reason})")
        continue
    diff = agg_mean - gridded_mean
    pct = 100 * diff / gridded_mean if gridded_mean else float("nan")
    print(f"{info['label']:<35} {agg_mean:>12.3f} {gridded_mean:>12.3f} {diff:>12.3f} {pct:>7.2f}%")

## Why HRU-level patterns differ across sources

After area-weighted aggregation to HRUs, the HRU-level magnitudes are within the same order of magnitude as the gridded means (validation cell above). The validation cell's "gridded" column is an unweighted mean over the consolidated-NC bbox, which extends past the fabric into ocean and Canada/Mexico edges; the "aggregated" column is an Albers-area-weighted mean over fabric HRUs only. A 5–30% difference between the two columns is normal for sources with out-of-fabric coverage. Differences above and beyond that geographic mismatch are physical:

- **SSEBop** is energy-balance-derived from satellite LST and meteorological forcing. It captures water stress directly through the surface energy balance. Native mm/month.
- **MOD16A2** is the MODIS Penman-Monteith global ET product. It uses VIIRS/MODIS LAI and surface reflectance with reanalysis forcing. The 8-day composite cadence smooths sub-monthly variability that SSEBop captures.
- **Coverage.** MOD16A2's 500 m native resolution means its NaN-HRU count after aggregation should be smaller than SSEBop's 1 km — confirmed by the coverage diagnostic above.

**Calibration target implication.** The AET target uses both sources as a per-HRU per-month `min/max`. The MOD16A2 `scale_factor=0.1` is the gotcha — without it, MOD16A2 reads 500× too high and the multi-source range becomes degenerate.

In [ ]:
for source_key, (ds, _) in opened.items():
    ds.close()
opened.clear()